# Projet 3: Moteur de recherche visuel sur catalogue de vêtements

Ce notebook documente la création d'un moteur de recherche visuel basé sur la similarité cosinus appliqué à des descripteurs d'images. Nous explorons les distributions du catalogue d'images téléchargé (CODAIT clothing-dataset), visualisons des échantillons d'images, étudions le descripteur combiné Couleur + Texture spatiale (histogramme couleur RVB et pixels de texture) et évaluons l'algorithme de recherche visuelle.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style="darkgrid")
print("Librairies prêtes.")

## 1. Chargement et Statistiques descriptives du Catalogue

In [ ]:
csv_path = "dataset/clothing-dataset/images.csv"
df = pd.read_csv(csv_path)
print(f"Catalogue chargé : {len(df)} images répertoriées.")
df.head()

In [ ]:
# Distribution des labels de vêtements dans le sous-ensemble actuel
plt.figure(figsize=(12, 6))
sns.countplot(data=df, y='label', order=df['label'].value_counts().index, palette='magma')
plt.title("Nombre de vêtements par type dans le catalogue de recherche")
plt.xlabel("Nombre d'images")
plt.ylabel("Catégorie")
plt.show()

## 2. Visualisation d'Échantillons d'Images

In [ ]:
# Afficher un échantillon de 4 images avec leurs catégories
sample_df = df.sample(4, random_state=12)

fig, axes = plt.subplots(1, 4, figsize=(15, 5))
for i, (idx, row) in enumerate(sample_df.iterrows()):
    img_path = os.path.join("dataset/clothing-dataset/images", row['image'] + ".jpg")
    if os.path.exists(img_path):
        img = Image.open(img_path)
        axes[i].imshow(img)
        axes[i].set_title(row['label'])
        axes[i].axis('off')
plt.tight_layout()
plt.show()

## 3. Analyse du Descripteur Couleur + Texture

In [ ]:
# Fonction pour illustrer l'histogramme couleur de l'image
sample_row = df.iloc[0]
img_path = os.path.join("dataset/clothing-dataset/images", sample_row['image'] + ".jpg")

if os.path.exists(img_path):
    img = Image.open(img_path).convert('RGB')
    pixels = np.array(img)
    
    fig, (ax_img, ax_hist) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax_img.imshow(img)
    ax_img.set_title(f"Image d'exemple : {sample_row['label']}")
    ax_img.axis('off')
    
    # Tracer les histogrammes couleur
    colors = ('r', 'g', 'b')
    for i, color in enumerate(colors):
        hist, bin_edges = np.histogram(pixels[:, :, i], bins=256, range=(0, 256))
        ax_hist.plot(bin_edges[0:-1], hist, color=color, alpha=0.8, label=f"Canal {color.upper()}")
        
    ax_hist.set_title("Histogramme couleur de l'image (R, G, B)")
    ax_hist.set_xlabel("Intensité pixel")
    ax_hist.set_ylabel("Densité de pixels")
    ax_hist.legend()
    plt.show()